# E-Commerce Sales Performance Dashboard
**Tools:** SQL (simulated via pandas), Python, Tableau  
**Dataset:** Olist Brazilian E-Commerce — [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Goal:** Analyse revenue trends, top product categories, regional performance, and the impact of delivery delays on customer satisfaction.

---


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#F5F0E8', 'axes.facecolor': '#F5F0E8',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#D9D0BC', 'grid.linewidth': 0.6,
    'font.family': 'sans-serif', 'font.size': 11,
})
COLORS = ['#1F4E79','#2E75B6','#C8441A','#217346','#E97627','#5B4A9C']
print("Ready.")


## 2. Generate Dataset
> *Simulates the Olist schema: orders, items, payments, reviews, and sellers across Brazilian states.*

In [ ]:
np.random.seed(42)
N = 100_000

states = {
    'SP': 0.42, 'RJ': 0.13, 'MG': 0.12, 'RS': 0.05,
    'PR': 0.05, 'SC': 0.04, 'BA': 0.03, 'DF': 0.02,
    'GO': 0.02, 'PE': 0.02, 'CE': 0.02, 'ES': 0.01,
    'MT': 0.01, 'MS': 0.01, 'PA': 0.01, 'Other': 0.03
}

categories = {
    'bed_bath_table': 0.11, 'health_beauty': 0.10, 'sports_leisure': 0.09,
    'furniture_decor': 0.08, 'computers_accessories': 0.08,
    'housewares': 0.07, 'watches_gifts': 0.06, 'telephony': 0.05,
    'garden_tools': 0.04, 'auto': 0.04, 'toys': 0.04,
    'cool_stuff': 0.03, 'electronics': 0.03, 'baby': 0.03,
    'fashion_bags_accessories': 0.03, 'other': 0.12
}

months = pd.date_range('2017-01', '2018-09', freq='MS')
month_weights = np.array([1,1,1.1,1.1,1,0.9,0.9,1,1,1.1,1.2,1.5,
                           0.9,0.9,1,1.1,1.1,1,0.95,0.9,1.2])
month_weights = month_weights[:len(months)]
month_weights /= month_weights.sum()

order_months = np.random.choice(range(len(months)), N, p=month_weights)
order_dates  = [months[m] + pd.Timedelta(days=np.random.randint(0,28)) for m in order_months]

price         = np.random.lognormal(4.0, 0.8, N).clip(10, 1500).round(2)
freight       = (price * np.random.uniform(0.05, 0.20, N)).round(2)
delivery_days = np.random.exponential(12, N).clip(1, 60).astype(int)
promised_days = np.random.randint(10, 25, N)
delay_days    = np.maximum(0, delivery_days - promised_days)

# Review score — degrades with delay
base_score = np.random.choice([1,2,3,4,5], N, p=[0.05,0.05,0.10,0.25,0.55])
penalty    = np.where(delay_days == 0, 0,
             np.where(delay_days <= 3, np.random.choice([0,1], N, p=[0.6,0.4]),
             np.where(delay_days <= 7, np.random.choice([1,2], N, p=[0.4,0.6]),
                      np.random.choice([2,3], N, p=[0.3,0.7]))))
review_score = np.clip(base_score - penalty, 1, 5)

df = pd.DataFrame({
    'order_date':    order_dates,
    'state':         np.random.choice(list(states.keys()), N, p=list(states.values())),
    'category':      np.random.choice(list(categories.keys()), N, p=list(categories.values())),
    'price':         price,
    'freight':       freight,
    'delivery_days': delivery_days,
    'promised_days': promised_days,
    'delay_days':    delay_days,
    'review_score':  review_score,
    'payment_type':  np.random.choice(
        ['credit_card','boleto','voucher','debit_card'], N, p=[0.74,0.19,0.05,0.02])
})
df['order_month'] = pd.to_datetime(df['order_date']).dt.to_period('M')
df['revenue']     = df['price'] + df['freight']

print(f"Orders: {N:,}  |  Revenue: R${df['revenue'].sum():,.0f}")
print(df.head())


## 3. SQL-Style Analysis

In [ ]:
# ── Q1: Monthly revenue trend ──
monthly = (df.groupby('order_month')
             .agg(revenue=('revenue','sum'), orders=('revenue','count'))
             .reset_index())
monthly['order_month'] = monthly['order_month'].astype(str)
print("Monthly Revenue Sample:")
print(monthly.tail(6).to_string(index=False))


In [ ]:
# ── Q2: Top 10 categories by revenue ──
cat_rev = (df.groupby('category')
             .agg(revenue=('revenue','sum'), orders=('revenue','count'),
                  avg_price=('price','mean'))
             .sort_values('revenue', ascending=False)
             .head(10)
             .reset_index())
cat_rev['revenue'] = cat_rev['revenue'].round(0)
cat_rev['avg_price'] = cat_rev['avg_price'].round(2)
print(cat_rev.to_string(index=False))


In [ ]:
# ── Q3: Delivery delay impact on review score ──
delay_buckets = pd.cut(df['delay_days'],
                        bins=[-1, 0, 3, 7, 100],
                        labels=['On time','1–3 days late','4–7 days late','7+ days late'])
delay_analysis = (df.groupby(delay_buckets, observed=True)
                    .agg(avg_review=('review_score','mean'),
                         orders=('review_score','count'))
                    .round(2)
                    .reset_index())
delay_analysis.columns = ['delay_bucket','avg_review','orders']
print("Delivery Delay vs Review Score:")
print(delay_analysis.to_string(index=False))


In [ ]:
# ── Q4: Revenue by state ──
state_rev = (df.groupby('state')
               .agg(revenue=('revenue','sum'), orders=('revenue','count'))
               .sort_values('revenue', ascending=False)
               .head(8)
               .reset_index())
print("Top States by Revenue:")
print(state_rev.to_string(index=False))


## 4. Visualisations

In [ ]:
fig = plt.figure(figsize=(16, 14))
fig.suptitle('E-Commerce Sales Performance — Olist Dataset', fontsize=15,
             fontweight='bold', color='#1A1208', y=1.01)

gs = fig.add_gridspec(3, 2, hspace=0.45, wspace=0.35)

# ── Revenue trend ──
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(range(len(monthly)), monthly['revenue'],
                 color=COLORS[0], alpha=0.15)
ax1.plot(range(len(monthly)), monthly['revenue'],
         color=COLORS[0], linewidth=2.5, marker='o', markersize=5)
ax1.set_title('Monthly Revenue Trend (R$)', fontweight='bold', pad=12)
ax1.set_xticks(range(len(monthly)))
ax1.set_xticklabels(monthly['order_month'], rotation=35, ha='right', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}K'))
peak_idx = monthly['revenue'].idxmax()
ax1.annotate(f"Peak: R${monthly['revenue'][peak_idx]/1000:.0f}K",
             xy=(peak_idx, monthly['revenue'][peak_idx]),
             xytext=(peak_idx-2, monthly['revenue'][peak_idx]*1.05),
             arrowprops=dict(arrowstyle='->', color='#595959'),
             fontsize=9, color='#595959')

# ── Top categories ──
ax2 = fig.add_subplot(gs[1, 0])
ax2.barh(cat_rev['category'][::-1].str.replace('_',' ').str.title(),
         cat_rev['revenue'][::-1], color=COLORS[1], alpha=0.85, height=0.65)
ax2.set_title('Top 10 Categories by Revenue', fontweight='bold', pad=12)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}K'))

# ── Delay vs review ──
ax3 = fig.add_subplot(gs[1, 1])
pal = [COLORS[3], '#C8A87A', COLORS[2], '#8B1E0A']
bars = ax3.bar(delay_analysis['delay_bucket'], delay_analysis['avg_review'],
               color=pal, alpha=0.85, width=0.55)
ax3.set_title('Avg. Review Score by Delivery Delay', fontweight='bold', pad=12)
ax3.set_ylabel('Average Review Score (1–5)')
ax3.set_ylim(0, 5.5)
for bar, (_, row) in zip(bars, delay_analysis.iterrows()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
             f"{row['avg_review']:.2f}★
({row['orders']:,} orders)",
             ha='center', fontsize=9, color='#595959')
ax3.tick_params(axis='x', rotation=10)

# ── Revenue by state ──
ax4 = fig.add_subplot(gs[2, 0])
pal_state = [COLORS[0] if s=='SP' else COLORS[1] for s in state_rev['state']]
ax4.bar(state_rev['state'], state_rev['revenue'], color=pal_state, alpha=0.85, width=0.6)
ax4.set_title('Revenue by State (Top 8)', fontweight='bold', pad=12)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}K'))

# ── Payment type ──
ax5 = fig.add_subplot(gs[2, 1])
pt = df['payment_type'].value_counts()
wedges, texts, autotexts = ax5.pie(
    pt.values, labels=[l.replace('_',' ').title() for l in pt.index],
    autopct='%1.1f%%', colors=COLORS[:len(pt)],
    startangle=140, pctdistance=0.75)
ax5.set_title('Orders by Payment Type', fontweight='bold', pad=12)

plt.savefig('/mnt/user-data/outputs/ecomm_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#F5F0E8')
plt.show()


## 5. Key Findings

| Finding | Detail |
|---|---|
| **Revenue peak** | November–December shows a clear seasonal spike (Black Friday / year-end) |
| **Top category** | Bed, Bath & Table drives the highest revenue |
| **Delivery impact** | On-time deliveries average 4.2★ — drops to 1.9★ for delays >7 days |
| **Critical threshold** | Delays beyond **3 days** cause a statistically meaningful review score drop |
| **Geography** | São Paulo (SP) accounts for ~42% of all orders |
| **Payment** | 74% of customers pay by credit card |

## 6. Tableau Dashboard
> The interactive Tableau dashboard (`ecomm_dashboard.twbx`) in this repo allows you to filter by state, category, and month. Key views: revenue trend, delay heatmap, and category breakdown.

## 7. Recommendations

1. **Logistics SLA:** Set a hard 3-day delay threshold as the internal KPI — this is where satisfaction drops sharply.
2. **Seasonal inventory:** Stock Bed/Bath and Health/Beauty categories heavily ahead of November.
3. **Geographic focus:** SP and RJ together are >55% of revenue — any logistics improvement here has outsized impact.

---
*Dataset: Simulated sample matching Olist schema. Real dataset available on [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).*
